# Clase 185 — A/B testing: tamaño de muestra y poder

Diseñamos y analizamos un A/B test end-to-end: `n` requerido para un MDE dado, análisis con z-test de proporciones + IC, el problema del **peeking**, la reducción de varianza con **CUPED** y una versión **bayesiana** con Beta.

Requiere: `numpy`, `scipy`, `statsmodels`, `matplotlib`.

## 1. Tamaño de muestra

`n` depende de α, poder (1-β) y del effect size. Para proporciones usamos **Cohen's h** y `NormalIndPower`.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

p0, p1 = 0.10, 0.11
h = proportion_effectsize(p1, p0)   # Cohen's h
n_req = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.8, alternative="two-sided")
print(f"Cohen's h = {h:.4f}")
print(f"n por grupo para detectar 10%→11% (poder 0.8, α=0.05) = {np.ceil(n_req):.0f}")
assert n_req > 5000

## 2. Análisis clásico: z-test de proporciones

Simulamos el experimento y reportamos p-value + IC95 % de la diferencia.

In [ ]:
nA = nB = 8000
convA = int(rng.binomial(nA, 0.10))
convB = int(rng.binomial(nB, 0.118))
stat, p = proportions_ztest([convB, convA], [nB, nA])
ci = confint_proportions_2indep(convB, nB, convA, nA, method="wald")
print(f"conv A={convA/nA:.4f}  B={convB/nB:.4f}")
print(f"z={stat:.3f}  p={p:.4f}  IC95% diferencia=({ci[0]:.4f}, {ci[1]:.4f})")
assert p < 0.05

## 3. CUPED: reducción de varianza con covariable pre

Ajustando por una covariable pre-experimento `X` correlacionada con `Y`, se reduce la varianza y sube el poder sin más muestra.

In [ ]:
n = 5000
X = rng.normal(50, 10, n)           # covariable pre
treat = rng.integers(0, 2, n)
Y = X + rng.normal(0, 5, n) + 2.0 * treat

theta = np.cov(Y, X)[0, 1] / np.var(X)
Y_cuped = Y - theta * (X - X.mean())

def welch_p(y, t):
    return stats.ttest_ind(y[t == 1], y[t == 0], equal_var=False).pvalue

print(f"var(Y)={Y.var():.1f}  var(Y_cuped)={Y_cuped.var():.1f}  reducción={1 - Y_cuped.var()/Y.var():.1%}")
print(f"p sin CUPED={welch_p(Y, treat):.4f}   con CUPED={welch_p(Y_cuped, treat):.2e}")
assert Y_cuped.var() < Y.var()

## 4. El problema del peeking

Mirar el resultado repetidamente y parar al primer `p < 0.05` infla α mucho más allá de 0.05, aunque `H₀` sea verdadera.

In [ ]:
reps = 1000
checkpoints = range(200, 5001, 200)
stopped_early = 0
for _ in range(reps):
    a = rng.normal(0, 1, 5000)
    b = rng.normal(0, 1, 5000)   # H0 verdadera: sin efecto
    for cp in checkpoints:
        if stats.ttest_ind(a[:cp], b[:cp]).pvalue < 0.05:
            stopped_early += 1
            break
alpha_real = stopped_early / reps
print(f"α real con peeking = {alpha_real:.3f} (nominal 0.05)")
assert alpha_real > 0.12

## 5. A/B bayesiano

Con priors Beta(1,1), el posterior de cada tasa es Beta. Respondemos directamente `P(p_B > p_A)`.

In [ ]:
# A: 1000 visitas, 80 conversiones   B: 1000 visitas, 100 conversiones
sa = stats.beta(1 + 80, 1 + 920).rvs(200_000, random_state=rng)
sb = stats.beta(1 + 100, 1 + 900).rvs(200_000, random_state=rng)
prob_b_better = np.mean(sb > sa)
print(f"P(p_B > p_A) = {prob_b_better:.3f}")
print(f"uplift esperado = {np.mean(sb - sa):.4f}")
assert prob_b_better > 0.9

plt.figure(figsize=(6, 4))
plt.hist(sa, bins=100, alpha=0.5, density=True, label="A (posterior)")
plt.hist(sb, bins=100, alpha=0.5, density=True, label="B (posterior)")
plt.legend(); plt.title("Posteriores Beta de la tasa de conversión")
plt.tight_layout(); plt.show()

## Ejercicios

1. Recalculá `n` con `power.NormalIndPower().solve_power` para un MDE de 0.5 pp y observá cómo `n ∝ 1/MDE²`.
2. Simulá un SRM: conteos A/B de 48/52 con `n=10⁶` y aplicá un χ² sobre los conteos para detectarlo.
3. En el A/B bayesiano, calculá `P(p_B > p_A + 0.01)` (uplift mínimo relevante) además del simple `p_B > p_A`.

## Conclusiones

- Fijá α, poder y MDE **antes** del experimento; `n` sale de ahí (y crece con `1/MDE²`).
- Reportá siempre effect size (Cohen's h) + IC, no solo el p-value.
- El peeking sin corrección infla α a ~0.2-0.3; usá sequential testing o pre-registrá el fin.
- CUPED reduce varianza con covariables pre; el A/B bayesiano da interpretación directa (`P(B > A)`).